In [77]:
import os
import pandas as pd

### RUTAS

Se realiza auditoria de los archivos, esto con el fin de revisar si es viable realizar la union de los registros de los datasets.

In [78]:
ruta = "/home/jovyan/work/data"
RUTA_ARCHIVOS = {}

for archivo in os.listdir(ruta):
    if archivo.endswith(".xlsx"):
        path = os.path.join(ruta, archivo)
        # leer solo encabezados (mucho más rápido)
        df = pd.read_excel(path, nrows=0)
        anio = archivo.split(".")[0]
        RUTA_ARCHIVOS[anio] = path
        print(f"Año {anio} -> {len(df.columns)} columnas")

Año 2014 -> 33 columnas
Año 2015 -> 33 columnas
Año 2016 -> 33 columnas
Año 2017 -> 33 columnas
Año 2018 -> 33 columnas
Año 2019 -> 39 columnas
Año 2020 -> 39 columnas
Año 2021 -> 41 columnas
Año 2022 -> 41 columnas
Año 2023 -> 41 columnas
Año 2024 -> 41 columnas


Se puede evidenciar que el numero de columnas aumenta respecto al año, esto puede deberse a que la entidad aumente el numero de columnas o variables con el fin de que el registro sea un poco más informativo.<br/><br/>
Debido a esto, se puede evidenciar que:
- Los archivos de los años 2014, 2015, 2016, 2017 y 2018 cuentan con 33 columnas.
- Los archivos de los años 2019 y 2020 cuentan con 39 columnas.
- Los archivos de los años 2021, 2022, 2023 y 2024 cuentan con 41 columnas.

### MAPEO DE COLUMNAS → ESQUEMA UNIFICADO

In [79]:
COLUMNAS_FINALES = [
    "codigo_institucion",
    "institucion",
    "sector",
    "caracter",
    "dpto_ies",
    "codigo_snies_programa",
    "programa_academico",
    "nivel_academico",
    "nivel_formacion",
    "metodologia",
    "area_conocimiento",
    "nucleo_conocimiento",
    "dpto_programa",
    "mpio_programa",
    "sexo",
    "anio",
    "semestre",
    "matriculados",
]

# Para cada columna estándar, qué nombres puede tener en los archivos fuente
VARIANTES = {
    "codigo_institucion":   ["codigodelainstitucion"],
    "institucion":          ["instituciondeeducacionsuperiories"],
    "sector":               ["sectories"],
    "caracter":             ["caracteries"],
    "dpto_ies":             ["departamentodedomiciliodelaies"],
    "codigo_snies_programa":["codigosniesprograma","codigosniesdelprograma"],
    "programa_academico":   ["programaacademico"],
    "nivel_academico":      ["nivelacademico"],
    "nivel_formacion":      ["niveldeformacion"],
    "metodologia":          ["metodologia", "modalidad"],           # cambia en 2023-2024
    "area_conocimiento":    ["areadeconocimiento"],
    "nucleo_conocimiento":  ["nucleobasicodelconocimientonbc"],
    "dpto_programa":        ["departamentodeofertadelprograma"],
    "mpio_programa":        ["municipiodeofertadelprograma"],
    "genero":               ["sexo", "genero"],                     # cambia en 2014-2015
    "anio":                 ["año", "anio"],
    "semestre":             ["semestre"],
    "matriculados":         ["matriculados", "matriculados2014",
                             "matriculados2015", "matriculados2016",
                             "matriculados2017", "matriculados2018"],
}

In [80]:
def extraer_columnas(df: pd.DataFrame, anio: str) -> pd.DataFrame:
    """
    Para cada columna estándar busca la primera variante que exista
    en el dataframe, la extrae y la renombra.
    """
    seleccion = {}

    for col_final, variantes in VARIANTES.items():
        encontrada = next((v for v in variantes if v in df.columns), None)
        if encontrada:
            seleccion[encontrada] = col_final
        else:
            print(f"[{anio}] ⚠️  No encontrada: '{col_final}'")

    return df.rename(columns=seleccion)[list(seleccion.values())]

### NORMALIZAR COLUMNAS

In [81]:
#Funcion que permite estandarizar el nombre de las columnas
def normalizar_columnas(df:pd.DataFrame):
    df.columns = ( \
        df.columns \
        .str.strip() \
        .str.lower() \
        .str.replace("\n", "", regex=False) \
        .str.replace(" ", "", regex=False) \
        .str.replace("(", "",  regex=False) \
        .str.replace(")", "",  regex=False) \
        .str.replace("á", "a", regex=False) \
        .str.replace("é", "e", regex=False) \
        .str.replace("í", "i", regex=False) \
        .str.replace("ó", "o", regex=False) \
        .str.replace("ú", "u", regex=False) \
        .str.replace("año", "anio", regex=False)
    )
    return df

### CARGA, MAPEO Y UNION

In [82]:
dfs = []

for anio, path in sorted(RUTA_ARCHIVOS.items()):
    print(f"Cargando {anio}...", end=" ")
    df = pd.read_excel(path, dtype=str)
    df = normalizar_columnas(df)
    df = extraer_columnas(df, anio)
    print(f"->{len(df):>8,} filas | {df.shape[1]} cols")
    dfs.append(df)

df_total = pd.concat(dfs, ignore_index=True, sort=False)

Cargando 2014... ->  60,838 filas | 18 cols
Cargando 2015... ->  59,996 filas | 18 cols
Cargando 2016... ->  60,363 filas | 18 cols
Cargando 2017... ->  65,382 filas | 18 cols
Cargando 2018... ->  63,385 filas | 18 cols
Cargando 2019... ->  70,232 filas | 18 cols
Cargando 2020... ->  69,585 filas | 18 cols
Cargando 2021... ->  70,607 filas | 18 cols
Cargando 2022... ->  69,755 filas | 18 cols
Cargando 2023... ->  72,373 filas | 18 cols
Cargando 2024... ->  75,175 filas | 18 cols


### EXPORTAR

In [83]:
df_total.to_csv(ruta + "/data.csv", index=False, encoding="utf-8")
print(f"{len(df_total):,} filas × {df_total.shape[1]} columnas → data.csv")
print(df_total.dtypes)

737,691 filas × 18 columnas → data.csv
codigo_institucion       str
institucion              str
sector                   str
caracter                 str
dpto_ies                 str
codigo_snies_programa    str
programa_academico       str
nivel_academico          str
nivel_formacion          str
metodologia              str
area_conocimiento        str
nucleo_conocimiento      str
dpto_programa            str
mpio_programa            str
genero                   str
anio                     str
semestre                 str
matriculados             str
dtype: object
